# Arithmetic Data Types Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the Arithmetic Data Types kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [ ]:
from psiqdk.workbench import Qubits, QUInt, QInt, Qubrick

### Problem 1. Increment by 1

What is the simplest way to increment a number written as a binary bit string $a_0a_1...a_{N-1}$?

1. Increment the least significant bit $a_0$ modulo $2$: convert $0$ to $1$ and vice versa. We can do that using an X gate.
2. If the updated value of the least significant bit $a_0^\prime = 1$, we're done with the increment. However, if $a_0^\prime = 0$, we know that $a_0 = 1$, and incrementing it exceeded the maximum value that can be stored in a bit. This means that we need to carry the extra digit into the second least significant bit $a_1$ and increment it. We can do this conditional increment using a controlled X gate, with the least significant qubit as the control and the second least significant qubit as the target.
3. Again, if the updated value of the second least significant bit $a_1^\prime = 1$, we're done, but if $a_1^\prime = 0$, we need to carry the extra digit into the third least significant bit $a_2$. This time, we need to check that both least significant qubits $a_0^\prime$ and $a_1^\prime$ are $0$ to apply the controlled gate. (If we just check that the previous bit $a_1^\prime = 0$, we'll run into an error for numbers such as $0_00_11_2$, for which the least significant bit was incremented without overflow and $a_1^\prime = a_1$.)
4. We continue this pattern of incrementing each bit if all less significant bits are $0$ after their own updates, until we arrive to the most significant bit $a_{N-1}$, which needs to be incremented only if all other bits $a_0^\prime a_1^\prime ... a_{N-2}^\prime$ are $0$.

Note that all X gates in this solution are applied if the control qubits are all in the $\ket{0}$ state. In Workbench, we can express this using the negation of the control subregister as the condition. For example, the first controlled X gate will be written as `a[1].x(cond=~a[0])`.

In [ ]:
def increment_1(a: QUInt) -> None:
    a[0].x()
    for ind in range(1, len(a)):
        a[ind].x(cond=~a[:ind])

Most quantum computing libraries, including Workbench, use a different implementation of the naive increment. Instead of starting the increment with the least significant bit $a_0$ and working our way up, we can start with the most significant bit $a_{N-1}$ and work our way down.

In this case, we need to increment the current bit only if all less significant bits $a_k$ are $1$ - in this case, once all of them are incremented, the overflow will carry into the current bit. To do this, we can use controlled X gates as well, conditioned on all control qubits being in the $\ket{1}$ state. The last bit to increment will be the least significant bit $a_0$, which is incremented using a plain X gate.

In [ ]:
def increment(a: QUInt) -> None:
    for ind in range(len(a) - 1, 0, -1):
        a[ind].x(cond=a[:ind])
    a[0].x()

### Problem 2. Increment by a power of 2

Again, let's consider the number $a$ and its little-endian notation $a_0 a_1...a_{p-1} a_p a_{p+1}...a_{N-1}$. 
The matching binary notation of $2^p$ is $0_0 ... 0_{p-1} 1_p$.

When we add $2^p$ to $a$, the $p$ least significant digits $a_0, a_1, ..., a_{p-1}$ are not changed, since the digits added to them are all $0$. The first digit that is changed is $a_p$, and the more significant digits after it are updated following the same rules as in simple increment. This means that you can split the resulting state in two parts: 

1. The $p$ least significant digits of the result are the same as in the initial state:
   $$a_0^\prime a_1^\prime ... a_{p-1}^\prime = a_0 a_1...a_{p-1}$$

2. The remaining $N-p$ digits are the result of incrementing the remaining digits of the initial state by $1$:
   $$a_p^\prime a_{p+1}^\prime ... a_N^\prime = a_p a_{p+1}...a_N + 1$$

To implement the solution, you can reuse the solution to the previous problem, applying it to the $N-p$ most significant digits of the register `a` `a[p:]`.

In [ ]:
def increment_power(a: QUInt, p: int) -> None:
    increment(a[p:])

### Problem 3. Increment by a constant

One way to add the constant $b$ to the number $a$ is to represent $b$ as a sum of powers of $2$ and add each of these powers of $2$ to $a$ separately. (This is a very inefficient approach, but we're looking at naive addition without any algorithmic improvements.)

The following solution iterates over all bits in $b$ and, if the bit is $1$, adds the corresponding power of $2$ to $a$ using the solution to the previous problem.

In [ ]:
def increment_constant(a: QUInt, b: int) -> None:
    m = b.bit_length()  # The number of bits in b
    for ind in range(m):
        if b & (1 << ind):
            increment(a[ind:])

### Problem 4. Add two unsigned integers

This task repeats the previous one, but with one difference: this time the addend $b$ is a quantum integer rather than a classical one. In the previous problem we used the bits of $b$ in a classical conditional statement, incrementing $a$ if a bit was $1$. We can use the same approach here, replacing the classical condition with a quantum one.

The Qubrick skeleton provided in the problem description suggests the structure of the solution.

The first step is to implement the `_increment` method with a quantum control. The easiest way to do that is to append the condition register `cond` to the conditions used in each controlled X gate (and to add it as a sole condition in the last X gate which used to be uncontrolled).

The second step is to implement the `_compute` method. The solution is similar to the previous task, but replaces the classical `if` statement checking that a bit of $b$ is $1$ with passing `b[ind]` as the quantum control register to the `_increment` method.

In [ ]:
class NaiveAdd(Qubrick):
    def _increment(self, a: QUInt, cond: Qubits) -> None:
        """Increment a QUInt register with a quantum control."""
        for ind in range(len(a) - 1, 0, -1):
            a[ind].x(cond=a[:ind] | cond)
        a[0].x(cond=cond)
    
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        m = len(b)
        for ind in range(m):
            self._increment(a[ind:], b[ind])

### Problem 5. Subtract two unsigned integers

Subtraction is the inverse of addition. In quantum computing terms, we can say that subtraction is the adjoint, or dagger, of addition.

The straightforward way to implement subtraction would be to take the gates used in addition and write them all in reverse order. (The controlled X gates we use in the code are all self-adjoint, so we don't need to worry about taking adjoint of each individual gate too.) This would be fine for a tiny code snippet like the ones in this kata, but generally we don't want to reimplement adjoint of an operation if we already implemented the operation itself - this is extra work that opens avenues to introducing new errors. Conveniently, Workbench [uncomputation](https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Qubricks-Uncomputation.html) provides an easy way to apply adjoint variants of operations implemented as Qubricks. Passing `dagger=True` as an additional argument to the `_compute` method of a Qubrick applies the adjoint of the computation it implements.

In our case, we want to implement subtraction as its own class, inheriting it from `NaiveAdd` Qubrick from the previous problem. We can avoid reimplementing its `_compute` method by implementing its `__init__` method instead; it needs to call the `__init__` method of the parent class with the additional argument `dagger=True`. This will make sure that any `compute` calls of `NaiveSubtract` will run the adjoint of the matching `compute` call of `NaiveAdd`.

> This is the reason we implemented the solution to the previous task as a Qubrick. Workbench handles uncomputation automatically using the Qubricks framework and can't do the same for regular Python functions.

In [ ]:
class NaiveSubtract(NaiveAdd):
    def __init__(self, **kwargs):
        super().__init__(dagger=True, **kwargs)

### Problem 6. Negate the number

Let's consider how to multiply a negative number $a$ in two's complement by $-1$. (We'll ignore the special case of $-2^{N-1}$ for now.)

We know that for a negative number $a$ its most significant bit is $a_{N-1} = 1$. Then it can be written as follows:

$$a = -2^{N-1} + a_{N-2} 2^{N-2} + \cdots + a_2 2^2 + a_1 2^1 + a_0 2^0$$

$-a$ will be the negation of this expression:

$$-a = 2^{N-1} - a_{N-2} 2^{N-2} - \cdots - a_2 2^2 - a_1 2^1 - a_0 2^0$$

Let's split $2^{N-1}$ into a sum of smaller powers of $2$:

$$2^{N-1} = 2^{N-2} + 2^{N-3} + ... + 2^2 + 2^1 + 2^0 + 1$$

If we substitute this into the expression for $-a$ and group the same powers of $2$ together (with the exception of the last $1$), we'll get:

$$-a = (1 - a_{N-2}) \cdot 2^{N-2} + (1 - a_{N-3}) \cdot 2^{N-3} + \cdots + (1 - a_2) \cdot 2^2 + (1 - a_1) \cdot 2^1 + (1 - a_0) \cdot 2^0 + 1$$

You can see that we can arrive from $a$ to $-a$ by flipping each bit of its binary notation and then incrementing the result.

The same formula holds for non-negative values of $a$; the only exception is $a = -2^{N-2}$, which, as mentioned in the problem statement, is a special case for two's complement notation, since it turns into its own negation under this transformation (same as $a = 0$).

We can confirm this formula by looking at the binary notations of numbers and their negations in two's complement, starting with $0$ and incrementing the bit string representing the number on each step, for $N = 3$.

| Binary $a$ | Decimal $a$ | Decimal $-a$ | Binary $-a$ |
| ----- | ----- | ----- | ----- |
| $000$ | $0$   | $0$   | $000$ |
| $100$ | $1$   | $-1$  | $111$ |
| $010$ | $2$   | $-2$  | $011$ |
| $110$ | $3$   | $-3$  | $101$ |
| $001$ | $-4$  | $-4$  | $001$ |
| $101$ | $-3$  | $3$   | $110$ |
| $011$ | $-2$  | $2$   | $010$ |
| $111$ | $-1$  | $1$   | $100$ |

To implement the solution, we can reuse the `increment` function from the first task; it was written for unsigned integers, for which increment behaves exactly the same as incrementing bit strings.

In [ ]:
def negate(a: QInt) -> None:
    a.x()
    increment(a)

### Problem 7. Add two signed integers of the same size

How can we add two numbers accounting for the fact that one or both of them can be negative?

It turns out that the logic of adding two signed integers in two's complement notation is exactly the same as that of adding unsigned integers! Indeed, an $N$-bit number $a$ in two's complement notation is the same as its unsigned counterpart, except if its value is greater than $2^{N-1} - 1$, it wraps around to indicate a negative value instead:

$$a_{signed} = \begin{cases}
a_{unsigned} \textrm{ if } 0 \le a_{unsigned} \le 2^{N-1} - 1 \\ 
a_{unsigned} - 2^{N} \textrm{ if } a_{unsigned} \ge 2^{N-1}
\end{cases}
$$

If we add two signed numbers $a_{signed}$ and $b_{signed}$, their sum will be the sum of $a_{unsigned}$ and $b_{unsigned}$, minus a multiple of $2^{N}$ which depends on whether the original numbers were positive or negative. But any multiples of $2^{N}$ will be discarded when the sum is converted to an $N$-bit number - there are no bits to store them. This means that we get the result we want automatically, by just repeating the code from problem 4!

In [ ]:
class NaiveAddSigned(Qubrick):
    def _increment(self, a: QInt, cond: Qubits) -> None:
        """Increment a QInt register with a quantum control."""
        for ind in range(len(a) - 1, 0, -1):
            a[ind].x(cond=a[:ind] | cond)
        a[0].x(cond=cond)

    def _compute(self, a: QInt, b: QInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        m = len(b)
        for ind in range(m):
            self._increment(a[ind:], b[ind])

### Problem 8. Add two signed integers of different sizes

We can't just reuse the same code for adding a shorter unsigned integer $b$ to a longer one $a$, though. If we try to do that, we'll be effectively adding bit strings $a_0a_1...a_{N-1}$ and $b_0b_1...b_{M-1}0...0$, which was fine for unsigned integers, but changes the sign of $b$ when it is treated as signed.

What we need to do instead is to figure out a bit string that represents the same value as $b_0b_1...b_{M-1}$ in $N$ bits regardless of the value of the sign bit $b_{M-1}$. This technique is called _sign extension_.

Let's discard the $M-1$ least significant bits (their values are fixed and don't affect what we do with the most significant bits). The most significant bit of $b$ on its own encodes the value $-b_{M-1} 2^{M-1}$, or, with the $M-1$ least significant digits discarded, just $-b_{M-1}$ - $0$ or $-1$. To perform sign extension, we want to write this value as a bit string with $N-M$ bits, and we know how to spell this out: $0...0$ and $1...1$, respectively!

This means that we need to take the most significant bit of the number $b$ and replicate it $N-M$ times to pad the number $b$ to $N$ bits. We don't need to allocate auxiliary qubits for this: we're only using the qubits of $b$ as controls for incrementing $a$, so we can just use the most significant qubit of $b$ for the last $N-M$ increments.

In [ ]:
class NaiveAddSignedExtension(NaiveAddSigned):
    def _compute(self, a: QInt, b: QInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        n = len(a)
        m = len(b)
        for ind in range(m):
            self._increment(a[ind:], b[ind])
        for ind in range(m, n):
            self._increment(a[ind:], b[m - 1])

> Copyright (c) 2026 PsiQuantum